# MOMENTA-Ozon: Детекция контрафакта с Mixture-of-Experts и мультимодальными эмбеддингами

**Адаптация архитектуры MOMENTA** *(Abdollahinejad et al., 2026)* под задачу выявления контрафакта на маркетплейсе Ozon.



## Инсайты из EDA, учтённые в архитектуре:
- **item_time_alive** и **seller_time_alive** — сильнейшие предикторы → temporal aggregation по истории продавца
- Несоответствие описания и изображения → discrepancy branch
- Сильный дисбаланс 93.4% / 6.6% → focal loss + weighted sampler
- NaN-флаги коррелируют с таргетом → явное моделирование пропусков
- Seller-level паттерны → EMA prototype memory bank по продавцам

## 0. Зависимости

In [ ]:
%%capture
!pip install torch transformers scikit-learn pandas numpy matplotlib seaborn pyarrow

## 1. Импорты и конфигурация

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from transformers import AutoTokenizer, AutoModel
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_recall_curve, classification_report
)
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
BERT_MODEL   = "DeepPavlov/rubert-base-cased"
MAX_LEN      = 128
BERT_BATCH   = 64
EMB_DIM      = 128
NUM_EXPERTS  = 4
NUM_HEADS    = 4
TEMP_WINDOW  = 8
TEMP_STRIDE  = 4
KAPPA        = 0.5
BETA_MOM     = 0.9
EMA_MOM      = 0.99
EPOCHS       = 20
LR           = 2e-4
WEIGHT_DECAY = 1e-2
BATCH_SIZE   = 256
DROPOUT      = 0.3
LAMBDA_ALIGN      = 0.05
LAMBDA_TC         = 0.01
LAMBDA_CONTRAST   = 0.05
LAMBDA_DOMAIN     = 0.03
LAMBDA_PROTO      = 0.05
LAMBDA_PROTO_MEM  = 0.10
LAMBDA_RDROP      = 0.5
TAU               = 0.2
GAMMA_FOC         = 2.5
LABEL_SMOOTH      = 0.05
MARGIN_PROTO      = 0.3

Device: cuda
Конфигурация MOMENTA-Ozon загружена


## 2. Загрузка данных

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd
import numpy as np
df = pd.read_csv('/content/drive/MyDrive/ozon_train.csv', encoding='utf-8')
print(f'Данные: {df.shape}')
print(f'Доля контрафакта: {df["resolution"].mean():.4f}')
clip_emb = pd.read_parquet('/content/drive/MyDrive/clip_embeddings.parquet')
clip_emb['vector'] = clip_emb['embedding'].apply(lambda x: np.array(x, dtype=np.float32))
print(f'CLIP embeddings: {clip_emb.shape}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Данные: (197198, 45)
Доля контрафакта: 0.0662
CLIP embeddings: (187604, 3)


## 3. Seller-based разбиение

EDA (sec 3.3): при random split Recall@P≥0.9 завышается ~в 15 раз из-за entity-level leakage.  
Используем GroupShuffleSplit по `SellerID`.

In [ ]:
seller_class_counts = df.groupby('SellerID')['resolution'].nunique()
multi_class_sellers = seller_class_counts[seller_class_counts > 1].index
np.random.seed(SEED)
test_sellers  = np.random.choice(multi_class_sellers,
size=int(0.3 * len(multi_class_sellers)),
replace=False)
train_sellers = np.setdiff1d(df['SellerID'].unique(), test_sellers)

train_df = df[df['SellerID'].isin(train_sellers)].copy().reset_index(drop=True)
test_df  = df[df['SellerID'].isin(test_sellers)].copy().reset_index(drop=True)

print(f'Train: {train_df.shape}, контрафакт: {train_df["resolution"].mean():.4f}')
print(f'Test:  {test_df.shape},  контрафакт: {test_df["resolution"].mean():.4f}')

Train: (177380, 45), контрафакт: 0.0588
Test:  (19818, 45),  контрафакт: 0.1321


## 4. Подготовка признаков

### 4.1 Seller LOO target encoding

EDA (sec 2.4): `seller_problem_rate` — один из сильнейших предикторов.  
LOO encoding на train предотвращает target leakage.

In [ ]:
from sklearn.model_selection import KFold
global_mean = train_df['resolution'].mean()
seller_loo = np.full(len(train_df), global_mean)
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
for _, (tr_idx, val_idx) in enumerate(kf.split(train_df)):
    fold_rate = train_df.iloc[tr_idx].groupby('SellerID')['resolution'].mean()
    seller_loo[val_idx] = train_df.iloc[val_idx]['SellerID'].map(fold_rate).fillna(global_mean).values
train_df['seller_problem_rate'] = seller_loo
seller_global = train_df.groupby('SellerID')['resolution'].mean().reset_index()
seller_global.columns = ['SellerID', 'seller_problem_rate']
test_df = test_df.merge(seller_global, on='SellerID', how='left')
test_df['seller_problem_rate'] = test_df['seller_problem_rate'].fillna(global_mean)
print('LOO encoding готов')

LOO encoding готов


### 4.2 Tabular feature engineering

На основе EDA:
- **NaN-флаги** (sec 2.5): факт пропуска коррелирует с таргетом
- **Ratio features** (sec 2.3): fake_return_rate информативнее абсолютных значений
- **Log-трансформации** (sec 1.6): убирают правый скос item_time_alive / seller_time_alive
- **Временные взаимодействия** (sec 2.6): произведение возрастов продавца и товара

In [ ]:
def engineer_tabular(df_):
    out = df_.copy()
    nan_flag_cols = ['brand_name', 'description',
                     'GmvTotal7', 'GmvTotal30', 'GmvTotal90', 'rating_5_count']
    for col in nan_flag_cols:
        if col in out.columns:
            out[f'is_null_{col}'] = out[col].isnull().astype(np.float32)
    for w in [7, 30, 90]:
        sales = out.get(f'item_count_sales{w}', pd.Series(0, index=out.index))
        ret   = out.get(f'item_count_returns{w}', pd.Series(0, index=out.index))
        fake  = out.get(f'item_count_fake_returns{w}', pd.Series(0, index=out.index))
        out[f'return_rate{w}']             = ret  / (sales + 1)
        out[f'fake_return_rate{w}']        = fake / (sales + 1)
        out[f'fake_to_all_returns{w}']     = fake / (ret   + 1)
    for col in ['item_time_alive', 'seller_time_alive', 'GmvTotal90',
                'rating_5_count', 'item_count_sales90']:
        if col in out.columns:
            out[f'log_{col}'] = np.log1p(out[col].fillna(0))
    if 'item_time_alive' in out.columns and 'seller_time_alive' in out.columns:
        out['interaction_age'] = (
            np.log1p(out['item_time_alive'].fillna(0)) *
            np.log1p(out['seller_time_alive'].fillna(0))
        )
    if 'SellerID' in out.columns and 'item_time_alive' in out.columns:
        out = out.sort_values(['SellerID', 'item_time_alive'])
        out['seller_item_rank'] = out.groupby('SellerID').cumcount().astype(np.float32)
    return out
train_df = engineer_tabular(train_df)
test_df  = engineer_tabular(test_df)
print(f'Признаков после FE: {train_df.shape[1]}')

Признаков после FE: 68


### 4.3 Текстовая модальность — ruBERT

In [ ]:
%%capture
tokenizer  = AutoTokenizer.from_pretrained(BERT_MODEL)
bert_model = AutoModel.from_pretrained(BERT_MODEL).to(device)
bert_model.eval()
print(f'ruBERT: {BERT_MODEL}')

BertModel LOAD REPORT from: DeepPavlov/rubert-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
bert.embeddings.position_ids               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
def make_text(df_):
    brand = df_['brand_name'].fillna('без бренда')
    return df_['name_rus'].fillna('') + ' ' + df_['description'].fillna('') + ' ' + brand
def get_bert_embeddings(texts, batch_size=BERT_BATCH, max_length=MAX_LEN):
    all_embs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size].tolist()
        encoded = tokenizer(
            batch, padding=True, truncation=True,
            max_length=max_length, return_tensors='pt'
        ).to(device)
        with torch.no_grad():
            out = bert_model(**encoded)
        emb = out.last_hidden_state[:, 0, :].cpu().numpy()
        all_embs.append(emb)
        if i % 10000 == 0:
            print(f'  {i}/{len(texts)}')
    return np.vstack(all_embs)
train_texts = make_text(train_df)
test_texts  = make_text(test_df)
X_train_text = get_bert_embeddings(train_texts)
X_test_text  = get_bert_embeddings(test_texts)
print(f'Text emb shape: {X_train_text.shape}')
TEXT_DIM = X_train_text.shape[1]

Извлекаем BERT-эмбеддинги...
  0/177380
  40000/177380
  80000/177380
  120000/177380
  160000/177380
  0/19818
Text emb shape: (177380, 768)


### 4.4 Визуальная модальность — CLIP embeddings

In [ ]:
train_clip = train_df[['ItemID', 'resolution']].merge(
    clip_emb[['ItemID', 'vector']], on='ItemID', how='inner')
test_clip  = test_df[['ItemID', 'resolution']].merge(
    clip_emb[['ItemID', 'vector']], on='ItemID', how='inner')
train_mask = train_df['ItemID'].isin(train_clip['ItemID']).values
test_mask  = test_df['ItemID'].isin(test_clip['ItemID']).values
y_tr = train_clip['resolution'].values.astype(np.float32)
y_te = test_clip['resolution'].values.astype(np.float32)
X_tr_text = X_train_text[train_mask]
X_te_text = X_test_text[test_mask]
X_tr_img = np.vstack(train_clip['vector'].values)
X_te_img = np.vstack(test_clip['vector'].values)
print(f'Train: {len(y_tr)}, контрафакт: {y_tr.mean():.4f}')
print(f'Test:  {len(y_te)}, контрафакт: {y_te.mean():.4f}')
IMG_DIM = X_tr_img.shape[1]

Train: 168619, контрафакт: 0.0586
Test:  18985, контрафакт: 0.1320


### 4.5 Табличные признаки

In [ ]:
drop_cols = {'id', 'ItemID', 'SellerID', 'name_rus', 'description',
             'brand_name', 'CommercialTypeName4', 'text', 'resolution'}
num_cols = [
    c for c in train_df.columns
    if c not in drop_cols and train_df[c].dtype in ['float64', 'int64', 'float32']
]
X_tr_tab = train_df[train_mask][num_cols].fillna(-1).values.astype(np.float32)
X_te_tab = test_df[test_mask][num_cols].fillna(-1).values.astype(np.float32)
TAB_DIM = X_tr_tab.shape[1]
print(f'Табличных признаков: {TAB_DIM}')
domain_col = 'CommercialTypeName4'
all_domains = train_df[domain_col].fillna('unknown').unique()
domain2id   = {d: i for i, d in enumerate(sorted(all_domains))}
NUM_DOMAINS = len(domain2id)
domain_tr = train_df[train_mask][domain_col].fillna('unknown').map(domain2id).fillna(0).values.astype(int)
domain_te  = test_df[test_mask][domain_col].fillna('unknown').map(domain2id).fillna(0).values.astype(int)
print(f'Доменов (категорий): {NUM_DOMAINS}')

Табличных признаков: 60
Доменов (категорий): 623


### 4.6 Нормализация

In [ ]:
scalers = {}
for name, tr, te in [
    ('text', X_tr_text, X_te_text),
    ('img',  X_tr_img,  X_te_img),
    ('tab',  X_tr_tab,  X_te_tab),
]:
    sc = StandardScaler()
    tr[:] = sc.fit_transform(tr)
    te[:] = sc.transform(te)
    scalers[name] = sc
print(f'Текст:    {X_tr_text.shape}')
print(f'Картинки: {X_tr_img.shape}')
print(f'Таблица:  {X_tr_tab.shape}')

Текст:    (168619, 768)
Картинки: (168619, 512)
Таблица:  (168619, 60)


## 5. Dataset и DataLoader

In [ ]:
class OzonDataset(Dataset):
    def __init__(self, text, img, tab, labels, domains, seller_ids, timestamps):
        self.text       = torch.tensor(text,       dtype=torch.float32)
        self.img        = torch.tensor(img,        dtype=torch.float32)
        self.tab        = torch.tensor(tab,        dtype=torch.float32)
        self.y          = torch.tensor(labels,     dtype=torch.float32)
        self.domain     = torch.tensor(domains,    dtype=torch.long)
        self.seller_ids = seller_ids
        self.timestamps = torch.tensor(timestamps, dtype=torch.float32)
    def __len__(self): return len(self.y)
    def __getitem__(self, idx):
        return (self.text[idx], self.img[idx], self.tab[idx],
                self.y[idx], self.domain[idx], self.timestamps[idx])
ts_tr = train_df[train_mask]['item_time_alive'].fillna(0).values.astype(np.float32)
ts_te = test_df[test_mask]['item_time_alive'].fillna(0).values.astype(np.float32)
sid_tr = train_df[train_mask]['SellerID'].values
sid_te = test_df[test_mask]['SellerID'].values
train_dataset = OzonDataset(X_tr_text, X_tr_img, X_tr_tab, y_tr, domain_tr, sid_tr, ts_tr)
test_dataset  = OzonDataset(X_te_text, X_te_img, X_te_tab, y_te, domain_te, sid_te, ts_te)
class_counts   = np.bincount(y_tr.astype(int))
class_weights  = 1.0 / class_counts
sample_weights = class_weights[y_tr.astype(int)]
sampler = WeightedRandomSampler(
    weights=torch.tensor(sample_weights, dtype=torch.float64),
    num_samples=len(sample_weights),
    replacement=True
)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          sampler=sampler, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2)
print(f'Train batches: {len(train_loader)}, Test batches: {len(test_loader)}')

Train batches: 659, Test batches: 75


## 6. Архитектура MOMENTA-Ozon

```
Текст (ruBERT, 768)    → Proj → LayerNorm → MHSA → MoE(K=4) → M_text (128)
                                                                              \
                                                                  Bidir        \
                                                               Co-Attention →  Gated Fusion Pi
                                                                              /         \
Картинка (CLIP, 512)   → Proj → LayerNorm → MHSA → MoE(K=4) → M_img  (128) /           \
                                                                              ↓            ↓
                                                                  Discrepancy  Temporal    Domain
                                                                  Branch       Aggregation  GRL
                                                                  di           Lw          ↓
                                                                               ↓        Domain
                                                                              pw         Pred
Таблица (~56)          → TabEncoder (→256→128) ─────────────────────────────/
                                                                   ↗ concat
                                                          [Lw ; ∆w ; Mw]
```

In [ ]:
class GradientReversalFn(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.clone()
    @staticmethod
    def backward(ctx, grad):
        return -ctx.alpha * grad, None
class GradientReversalLayer(nn.Module):
    def __init__(self):
        super().__init__()
        self.alpha = 0.0
    def forward(self, x):
        return GradientReversalFn.apply(x, self.alpha)
class Expert(nn.Module):
    def __init__(self, dim, expand=4, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim * expand),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim * expand, dim),
        )
    def forward(self, x):
        return self.net(x)
class MoELayer(nn.Module):
    def __init__(self, dim, num_experts=NUM_EXPERTS, dropout=0.1):
        super().__init__()
        self.experts = nn.ModuleList([Expert(dim, dropout=dropout) for _ in range(num_experts)])
        self.gate    = nn.Linear(dim, num_experts)
    def forward(self, x):
        alpha = F.softmax(self.gate(x), dim=-1)
        out   = torch.stack([e(x) for e in self.experts], dim=1)
        return (alpha.unsqueeze(-1) * out).sum(dim=1)
class ModalityEncoder(nn.Module):
    def __init__(self, input_dim, proj_dim=EMB_DIM, num_heads=NUM_HEADS,
                 num_experts=NUM_EXPERTS, dropout=DROPOUT):
        super().__init__()
        self.proj      = nn.Linear(input_dim, proj_dim)
        self.ln_proj   = nn.LayerNorm(proj_dim)
        self.mhsa      = nn.MultiheadAttention(proj_dim, num_heads,
                                               dropout=dropout, batch_first=True)
        self.ln_mhsa   = nn.LayerNorm(proj_dim)
        self.moe       = MoELayer(proj_dim, num_experts, dropout)
    def forward(self, x):
        h = self.ln_proj(self.proj(x))
        h = h.unsqueeze(1)
        attn_out, _ = self.mhsa(h, h, h)
        h = self.ln_mhsa(h + attn_out).squeeze(1)
        return self.moe(h)
class BidirCoAttention(nn.Module):
    def __init__(self, dim, num_heads=NUM_HEADS, dropout=DROPOUT):
        super().__init__()
        self.WQ = nn.Linear(dim, dim, bias=False)
        self.WK = nn.Linear(dim, dim, bias=False)
        self.WV = nn.Linear(dim, dim, bias=False)
        self.scale = dim ** -0.5
        self.ln_t  = nn.LayerNorm(dim)
        self.ln_i  = nn.LayerNorm(dim)
    def forward(self, mt, mi):
        qt, ki, vi = self.WQ(mt), self.WK(mi), self.WV(mi)
        qi, kt, vt = self.WQ(mi), self.WK(mt), self.WV(mt)
        a_ti = (qt * ki * self.scale).sum(-1, keepdim=True)
        c_ti = torch.sigmoid(a_ti) * vi
        a_it = (qi * kt * self.scale).sum(-1, keepdim=True)
        c_it = torch.sigmoid(a_it) * vt
        mt2 = self.ln_t(mt + c_ti)
        mi2 = self.ln_i(mi + c_it)
        return mt2, mi2

class TabEncoder(nn.Module):
    def __init__(self, tab_dim, out_dim=EMB_DIM, dropout=DROPOUT):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(tab_dim, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, out_dim),
            nn.LayerNorm(out_dim),
        )
    def forward(self, x):
        return self.net(x)

class MOMENTAOzon(nn.Module):
    def __init__(self, text_dim, img_dim, tab_dim, num_domains,
                 emb_dim=EMB_DIM, dropout=DROPOUT):
        super().__init__()
        d = emb_dim
        self.text_enc = ModalityEncoder(text_dim, d)
        self.img_enc  = ModalityEncoder(img_dim,  d)
        self.tab_enc  = TabEncoder(tab_dim, d, dropout)
        self.co_attn  = BidirCoAttention(d)
        self.Wg = nn.Linear(2 * d, d)
        self.Wdisc = nn.Linear(2 * d, d)
        self.ln_disc = nn.LayerNorm(d)
        self.eta = nn.Parameter(torch.zeros(1))
        self.match_head = nn.Linear(2 * d, 1)
        self.grl = GradientReversalLayer()
        self.domain_head = nn.Sequential(
            nn.Linear(d, 64), nn.ReLU(), nn.Linear(64, num_domains)
        )
        self.WV_temp = nn.Linear(d, d)
        self.WQ_temp = nn.Linear(d, d)
        self.classifier_temp = nn.Linear(2 * d + 1, 1)
        self.register_buffer('proto', torch.zeros(2, d))
        self.proto_initialized = [False, False]
        fusion_dim = d + d
        self.final_proj = nn.Linear(fusion_dim, d)
        self.ln_final   = nn.LayerNorm(d)
        self.head       = nn.Linear(d, 1)
        self.dropout = nn.Dropout(dropout)─
    def _fuse(self, mt2, mi2):
        gi = torch.sigmoid(self.Wg(torch.cat([mt2, mi2], dim=-1)))
        Pi = gi * mt2 + (1 - gi) * mi2
        d_abs  = (mt2 - mi2).abs()
        d_prod = mt2 * mi2
        di = self.ln_disc(self.Wdisc(torch.cat([d_abs, d_prod], dim=-1)))
        Pi = Pi + torch.tanh(self.eta) * di
        return Pi, gi
    def _alignment_score(self, mt2, mi2):
        return F.cosine_similarity(mt2, mi2, dim=-1)
    @torch.no_grad()
    def update_prototypes(self, Pi, labels, momentum=EMA_MOM):
        for c in [0, 1]:
            mask = (labels == c)
            if mask.sum() == 0:
                continue
            mu = F.normalize(Pi[mask].mean(0), dim=-1)
            if not self.proto_initialized[c]:
                self.proto[c] = mu
                self.proto_initialized[c] = True
            else:
                self.proto[c] = momentum * self.proto[c] + (1 - momentum) * mu
                self.proto[c] = F.normalize(self.proto[c], dim=-1)

    def prototype_loss(self, Pi, labels):
        if not all(self.proto_initialized):
            return torch.tensor(0.0, device=Pi.device)
        loss = 0.0
        for i in range(len(Pi)):
            yi  = int(labels[i].item())
            yni = 1 - yi
            sim_pos = F.cosine_similarity(Pi[i:i+1], self.proto[yi:yi+1], dim=-1)
            sim_neg = F.cosine_similarity(Pi[i:i+1], self.proto[yni:yni+1], dim=-1)
            loss += (1 - sim_pos) + F.relu(sim_neg - MARGIN_PROTO)
        return loss / len(Pi)

    def forward(self, text, img, tab, timestamps=None):
        Mt = self.text_enc(text)
        Mi = self.img_enc(img)
        Xt = self.tab_enc(tab)
        Mt2, Mi2 = self.co_attn(Mt, Mi)
        align_score = self._alignment_score(Mt2, Mi2)
        Pi, _ = self._fuse(Mt2, Mi2)
        z_match   = torch.cat([Mt2, Mi2], dim=-1)
        match_logit = self.match_head(z_match).squeeze(-1)
        domain_logit = self.domain_head(self.grl(Pi))
        fused = torch.cat([Pi, Xt], dim=-1)
        h     = self.ln_final(F.gelu(self.final_proj(fused)))
        h     = self.dropout(h)
        logit = self.head(h).squeeze(-1)

        return logit, align_score, match_logit, domain_logit, Pi
class TemporalAggregator(nn.Module):
    def __init__(self, emb_dim=EMB_DIM, window=TEMP_WINDOW, stride=TEMP_STRIDE,
                 kappa=KAPPA, beta=BETA_MOM):
        super().__init__()
        self.window  = window
        self.stride  = stride
        self.kappa   = kappa
        self.beta    = beta
        self.WV = nn.Linear(emb_dim, emb_dim, bias=False)
        self.WQ = nn.Linear(emb_dim, emb_dim, bias=False)
        self.clf = nn.Linear(2 * emb_dim + 1, 1)

    def forward(self, Pi, timestamps, labels):
        order = torch.argsort(timestamps)
        Pi_s  = Pi[order]
        ts_s  = timestamps[order]
        lb_s  = labels[order]

        N, d = Pi_s.shape
        t_max = ts_s.max().item()

        window_logits = []
        window_labels = []
        Lw_prev = None
        Mw      = torch.zeros(1, device=Pi.device)

        starts = list(range(0, max(1, N - self.window + 1), self.stride))
        for start in starts:
            end = min(start + self.window, N)
            Pi_w  = Pi_s[start:end]
            ts_w  = ts_s[start:end]
            lb_w  = lb_s[start:end]
            decay = torch.exp(-self.kappa * (t_max - ts_w) / (86400 + 1e-8))
            q  = self.WQ(Pi_w.mean(0, keepdim=True))
            k  = self.WK(Pi_w) if hasattr(self, 'WK') else self.WQ(Pi_w)
            attn = F.softmax((q @ k.T).squeeze(0) / (d ** 0.5), dim=0)
            combined = decay * attn
            combined = combined / (combined.sum() + 1e-8)

            v  = self.WV(Pi_w)
            Lw = (combined.unsqueeze(-1) * v).sum(0)
            if Lw_prev is None:
                delta = torch.zeros_like(Lw)
            else:
                delta = Lw - Lw_prev
            Mw = self.beta * Mw + (1 - self.beta) * delta.norm(p=2).unsqueeze(0)
            L_bar = torch.cat([Lw, delta, Mw])
            window_logits.append(self.clf(L_bar).squeeze())
            window_labels.append(lb_w[-1])
            Lw_prev = Lw.detach()

        if len(window_logits) == 0:
            return None, None

        return torch.stack(window_logits), torch.stack(window_labels)

Архитектура MOMENTA-Ozon определена


## 7. Multi-Objective Loss (Sec 3.9)

Комбинация из статьи: focal + align + contrastive + temporal_consistency + rdrop + domain + proto

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, gamma=GAMMA_FOC, alpha=0.80, smooth=LABEL_SMOOTH):
        super().__init__()
        self.gamma  = gamma
        self.alpha  = alpha
        self.smooth = smooth

    def forward(self, logits, targets):
        y_smooth = targets * (1 - self.smooth) + self.smooth / 2
        bce    = F.binary_cross_entropy_with_logits(logits, y_smooth, reduction='none')
        probs  = torch.sigmoid(logits)
        p_t    = probs * targets + (1 - probs) * (1 - targets)
        alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        return (alpha_t * (1 - p_t) ** self.gamma * bce).mean()


def alignment_loss(align_score):
    return (1 - align_score).mean()


def contrastive_loss(T, I, tau=TAU):
    T = F.normalize(T, dim=-1)
    I = F.normalize(I, dim=-1)
    logits = T @ I.T / tau   # (B, B)
    labels = torch.arange(len(T), device=T.device)
    lt = F.cross_entropy(logits, labels)
    li = F.cross_entropy(logits.T, labels)
    return (lt + li) / 2


def temporal_consistency_loss(Pi_list, rho=0.9):
    if len(Pi_list) < 2:
        return torch.tensor(0.0)
    loss = 0.0
    for w in range(1, len(Pi_list)):
        loss += (rho ** w) * (Pi_list[w] - Pi_list[w-1]).pow(2).mean()
    return loss


def rdrop_loss(p1, p2):
    p1 = torch.sigmoid(p1).clamp(1e-6, 1 - 1e-6)
    p2 = torch.sigmoid(p2).clamp(1e-6, 1 - 1e-6)
    kl1 = p1 * (p1 / p2).log() + (1 - p1) * ((1 - p1) / (1 - p2)).log()
    kl2 = p2 * (p2 / p1).log() + (1 - p2) * ((1 - p2) / (1 - p1)).log()
    return (kl1 + kl2).mean() / 2


focal_loss = FocalLoss()
print('Multi-Objective Loss определён')

Multi-Objective Loss определён


## 8. Инициализация модели

In [ ]:
model = MOMENTAOzon(
    text_dim    = TEXT_DIM,
    img_dim     = IMG_DIM,
    tab_dim     = TAB_DIM,
    num_domains = NUM_DOMAINS,
    emb_dim     = EMB_DIM,
    dropout     = DROPOUT,
).to(device)

temporal_agg = TemporalAggregator(
    emb_dim = EMB_DIM,
    window  = TEMP_WINDOW,
    stride  = TEMP_STRIDE,
).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Параметров MOMENTA-Ozon: {total_params:,}')

optimizer = AdamW(
    list(model.parameters()) + list(temporal_agg.parameters()),
    lr=LR, weight_decay=WEIGHT_DECAY
)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)

Параметров MOMENTA-Ozon: 1,632,508


## 9. Обучение

In [ ]:
def evaluate(model, loader, device):
    model.eval()
    temporal_agg.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for text, img, tab, y, domain, ts in loader:
            logit, _, _, _, _ = model(text.to(device), img.to(device), tab.to(device))
            all_probs.append(torch.sigmoid(logit).cpu().numpy())
            all_labels.append(y.numpy())
    y_pred = np.concatenate(all_probs)
    y_true = np.concatenate(all_labels)
    return roc_auc_score(y_true, y_pred), average_precision_score(y_true, y_pred)


best_pr_auc = 0.0
best_state  = None
history     = {'train_loss': [], 'val_roc': [], 'val_pr': [],
               'loss_focal': [], 'loss_align': [], 'loss_contrast': [],
               'loss_domain': [], 'loss_proto': []}

for epoch in range(1, EPOCHS + 1):
    model.train()
    temporal_agg.train()
    p = epoch / EPOCHS
    model.grl.alpha = 2.0 / (1 + np.exp(-10 * p)) - 1

    total_loss = 0.0
    lf = la = lc = ld = lp = 0.0

    for batch in train_loader:
        text, img, tab, y, domain, ts = batch
        text, img, tab = text.to(device), img.to(device), tab.to(device)
        y, domain, ts  = y.to(device), domain.to(device), ts.to(device)
        optimizer.zero_grad()
        logit, align_sc, match_logit, dom_logit, Pi = model(text, img, tab, ts)
        logit2, align_sc2, _, _, Pi2 = model(text, img, tab, ts)
        loss_focal = focal_loss(logit, y)
        loss_align = alignment_loss(align_sc)
        loss_contrast = contrastive_loss(Pi, Pi2)
        loss_tc = (Pi - Pi2).pow(2).mean()
        loss_rdrop = rdrop_loss(logit, logit2)
        loss_domain = F.cross_entropy(dom_logit, domain)
        mu0 = Pi[y == 0].mean(0) if (y == 0).any() else Pi.mean(0)
        mu1 = Pi[y == 1].mean(0) if (y == 1).any() else Pi.mean(0)
        loss_proto = torch.tensor(0.0, device=device)
        for i in range(len(Pi)):
            yi  = int(y[i].item())
            mu_pos = mu1 if yi == 1 else mu0
            mu_neg = mu0 if yi == 1 else mu1
            loss_proto += (1 - F.cosine_similarity(Pi[i:i+1], mu_pos.unsqueeze(0))).squeeze()
            loss_proto += F.relu(F.cosine_similarity(Pi[i:i+1], mu_neg.unsqueeze(0)) - MARGIN_PROTO).squeeze()
        loss_proto = loss_proto / len(Pi)
        model.update_prototypes(Pi.detach(), y)
        with torch.enable_grad():
            tw_logits, tw_labels = temporal_agg(Pi.detach(), ts, y)
        if tw_logits is not None:
            loss_temporal = focal_loss(tw_logits, tw_labels.float())
        else:
            loss_temporal = torch.tensor(0.0, device=device)
        loss = (loss_focal
                + LAMBDA_ALIGN    * loss_align
                + LAMBDA_TC       * loss_tc
                + LAMBDA_CONTRAST * loss_contrast
                + LAMBDA_RDROP    * loss_rdrop
                + LAMBDA_DOMAIN   * loss_domain
                + LAMBDA_PROTO    * loss_proto
                + LAMBDA_PROTO_MEM * loss_temporal)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()
        lf += loss_focal.item()
        la += loss_align.item()
        lc += loss_contrast.item()
        ld += loss_domain.item()
        lp += loss_proto.item()

    scheduler.step()
    val_roc, val_pr = evaluate(model, test_loader, device)
    n = len(train_loader)
    history['train_loss'].append(total_loss / n)
    history['val_roc'].append(val_roc)
    history['val_pr'].append(val_pr)
    history['loss_focal'].append(lf / n)
    history['loss_align'].append(la / n)
    history['loss_contrast'].append(lc / n)
    history['loss_domain'].append(ld / n)
    history['loss_proto'].append(lp / n)

    if val_pr > best_pr_auc:
        best_pr_auc = val_pr
        best_state  = {k: v.cpu() for k, v in model.state_dict().items()}

    print(f'Epoch {epoch:2d}/{EPOCHS} | '
          f'Loss={total_loss/n:.4f} (F={lf/n:.3f} A={la/n:.3f} C={lc/n:.3f} D={ld/n:.3f}) | '
          f'ROC={val_roc:.4f} PR={val_pr:.4f} | best={best_pr_auc:.4f}')

Epoch  1/20 | Loss=0.2957 (F=0.016 A=0.220 C=1.677 D=4.667) | ROC=0.8751 PR=0.5330 | best=0.5330
Epoch  2/20 | Loss=0.2831 (F=0.012 A=0.040 C=1.478 D=5.042) | ROC=0.8750 PR=0.5562 | best=0.5562
Epoch  3/20 | Loss=0.2716 (F=0.011 A=0.036 C=1.399 D=4.852) | ROC=0.8564 PR=0.5038 | best=0.5562
Epoch  4/20 | Loss=0.2697 (F=0.009 A=0.032 C=1.351 D=4.935) | ROC=0.8446 PR=0.4954 | best=0.5562
Epoch  5/20 | Loss=0.2674 (F=0.008 A=0.025 C=1.313 D=4.983) | ROC=0.8517 PR=0.4954 | best=0.5562
Epoch  6/20 | Loss=0.2617 (F=0.007 A=0.024 C=1.289 D=4.861) | ROC=0.8550 PR=0.5124 | best=0.5562
Epoch  7/20 | Loss=0.2580 (F=0.006 A=0.025 C=1.277 D=4.776) | ROC=0.8489 PR=0.4888 | best=0.5562
Epoch  8/20 | Loss=0.2568 (F=0.006 A=0.028 C=1.267 D=4.761) | ROC=0.8377 PR=0.4697 | best=0.5562
Epoch  9/20 | Loss=0.2555 (F=0.005 A=0.026 C=1.259 D=4.745) | ROC=0.8372 PR=0.4650 | best=0.5562
Epoch 10/20 | Loss=0.2543 (F=0.005 A=0.025 C=1.258 D=4.737) | ROC=0.8433 PR=0.4793 | best=0.5562
Epoch 11/20 | Loss=0.2537 (F=0

## 10. Итоговая оценка

In [ ]:
model.load_state_dict(best_state)
model.eval()

all_probs, all_labels = [], []
with torch.no_grad():
    for text, img, tab, y, domain, ts in test_loader:
        logit, _, _, _, _ = model(text.to(device), img.to(device), tab.to(device))
        all_probs.append(torch.sigmoid(logit).cpu().numpy())
        all_labels.append(y.numpy())

y_pred = np.concatenate(all_probs)
y_true = np.concatenate(all_labels)

roc = roc_auc_score(y_true, y_pred)
pr  = average_precision_score(y_true, y_pred)

precision_arr, recall_arr, thresholds_arr = precision_recall_curve(y_true, y_pred)
mask = precision_arr >= 0.9
recall_at_90 = recall_arr[mask].max() if mask.any() else 0.0

print(f'ROC-AUC:        {roc:.4f}')
print(f'PR-AUC:         {pr:.4f}')
print(f'Recall@P≥0.9:   {recall_at_90:.4f}')
print()
print(classification_report(y_true, (y_pred >= 0.5).astype(int),
                             target_names=['Легитимный', 'Контрафакт']))

## 11. Сравнение с предыдущими версиями

In [ ]:
results = pd.DataFrame([
    {'Модель': 'Baseline CatBoost',  'ROC-AUC': 0.9051, 'PR-AUC': 0.6231, 'Recall@P≥0.9': 0.0145},
    {'Модель': 'SpotFake-Ozon',      'ROC-AUC': 0.9134, 'PR-AUC': 0.6660, 'Recall@P≥0.9': 0.0000},
    {'Модель': 'SAFE-Ozon',       'ROC-AUC': 0.9131, 'PR-AUC': 0.6564, 'Recall@P≥0.9': 0.0012},
    {'Модель': 'Spotfake combined', 'ROC-AUC': 0.9168, 'PR-AUC': 0.6939, 'Recall@P≥0.9': 0.1265},
    {'Модель': 'MOMENTA-Ozon',       'ROC-AUC': roc,    'PR-AUC': pr,     'Recall@P≥0.9': recall_at_90},
]).set_index('Модель')

results['PR-AUC Δ vs SpotFake'] = (results['PR-AUC'] - 0.6660).round(4)

display(results.style
    .highlight_max(subset=['ROC-AUC', 'PR-AUC', 'Recall@P≥0.9'], color='lightgreen')
    .format('{:.4f}'))

## 12. Ablation Study: вклад компонентов MOMENTA

In [ ]:
def ablate_and_eval(model, loader, device, ablation_name,
                    zero_domain=False, zero_tab=False, skip_discrepancy=False):
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for text, img, tab, y, domain, ts in loader:
            text, img, tab = text.to(device), img.to(device), tab.to(device)

            if zero_tab:
                tab = torch.zeros_like(tab)

            if skip_discrepancy:
                orig_eta = model.eta.data.clone()
                model.eta.data.fill_(-10.0)

            logit, _, _, _, _ = model(text, img, tab, ts.to(device))

            if skip_discrepancy:
                model.eta.data.copy_(orig_eta)

            all_probs.append(torch.sigmoid(logit).cpu().numpy())
            all_labels.append(y.numpy())

    y_p = np.concatenate(all_probs)
    y_t = np.concatenate(all_labels)
    roc_ab = roc_auc_score(y_t, y_p)
    pr_ab  = average_precision_score(y_t, y_p)
    mask   = (precision_arr if hasattr(precision_arr, '__len__') else
              precision_recall_curve(y_t, y_p)[0]) >= 0.9
    print(f'{ablation_name:30s} | ROC={roc_ab:.4f} | PR={pr_ab:.4f}')
    return roc_ab, pr_ab

ablate_and_eval(model, test_loader, device, 'MOMENTA-Ozon (полная)')
ablate_and_eval(model, test_loader, device, 'Без табличных признаков', zero_tab=True)
ablate_and_eval(model, test_loader, device, 'Без Discrepancy Branch', skip_discrepancy=True)